conda install -c conda-forge scikit-optimize

In [ ]:
import os
import math

from skopt import gp_minimize
from skopt import Optimizer

### Read the census block extents.

In [ ]:
df = spark\
    .read\
    .parquet("/data/CensusBlocks.par")\
    .withColumnRenamed("EXT_MIN_X", "xmin")\
    .withColumnRenamed("EXT_MIN_Y", "ymin")\
    .withColumnRenamed("EXT_MAX_X", "xmax")\
    .withColumnRenamed("EXT_MAX_Y", "ymax")\
    .cache()

### Calculate extent widths and heights

In [ ]:
dxdy = df\
    .selectExpr("(xmax-xmin) dx","(ymax-ymin) dy")\
    .cache()

### Calculate avg cell size

In [ ]:
res = dxdy\
    .selectExpr("count(1) num_areas","sum(dx*dy) tot_areas")\
    .collect()

num_areas, tot_areas = res[0]
cell = math.sqrt(30.0 * tot_areas / num_areas)
cell

### Define function to calculate max count per cell and avg cell counts

In [ ]:
def f(x):
    x = x[0]
    df\
        .selectExpr(f"cast(floor(xmin/{x}) as long) q",f"cast(floor(ymin/{x}) as long) r")\
        .createOrReplaceTempView("v1")

    rows = sql("""
    select max(pop) max_pop,avg(pop) avg_pop
    from
    (
    select count(1) pop
    from v1
    group by q,r
    )
    """)\
        .collect()[0]

    n,a = rows
    print(f"n={n} a={a}")
    return 2.0 if n > 30 else 0.8*(1.0 - n / 30.0) + 1.2*(1.0 - a / 30.0)

In [ ]:
opt = Optimizer([(cell / 3.3, cell / 3.1)])

for i in range(30):
    suggested = opt.ask()
    y = f(suggested)
    opt.tell(suggested, y)
    print(i, suggested, y)